In [2]:
!pip install datasets -q

from datasets import load_dataset, Dataset, DatasetDict

print("Connecting to WMT14 dataset stream...")
# 1. Load the WMT14 English-French dataset using streaming to avoid massive downloads
wmt14_stream = load_dataset("wmt/wmt14", "fr-en", split="train", streaming=True)

print("Extracting a manageable subset of 25,000 rows...")
# 2. Extract exactly 25,000 rows from the stream
subset_list = list(wmt14_stream.take(25000))
subset_dataset = Dataset.from_list(subset_list)

# 3. Divide into training (80%), validation (10%), and test (10%) sets
train_test_split = subset_dataset.train_test_split(test_size=0.2, seed=42)
val_test_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

final_dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': val_test_split['train'],
    'test': val_test_split['test']
})

print("\nDataset successfully prepared:")
print(final_dataset)
print("\nExample parallel text:")
print(final_dataset['train'][0]['translation'])

Connecting to WMT14 dataset stream...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Extracting a manageable subset of 25,000 rows...

Dataset successfully prepared:
DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2500
    })
})

Example parallel text:
{'en': 'Let us therefore take great care with work in a closed environment.', 'fr': 'Soyons donc très attentifs aux travaux en milieu fermé.'}


In [4]:
!pip install transformers[torch] accelerate sacremoses -q

from transformers import AutoTokenizer, AutoConfig, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

print("Loading tokenizer and preparing data...")
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=64, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = final_dataset.map(preprocess_function, batched=True)

print("Initializing the Basic Transformer from scratch...")
config = AutoConfig.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model = AutoModelForSeq2SeqLM.from_config(config) 

training_args = Seq2SeqTrainingArguments(
    output_dir="./basic_transformer_results",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3, 
    predict_with_generate=True,
    fp16=True, 
    save_strategy="epoch",
    report_to="none" 
)

# 5. Train the model using the updated argument
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer, # Changed from 'tokenizer' to 'processing_class'
)

print("Starting training! This will take a few minutes...")
import time
start_time = time.time()

trainer.train()

training_time = time.time() - start_time
print(f"Training completed in {training_time / 60:.2f} minutes.")

Loading tokenizer and preparing data...


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed in 7.80 minutes.


In [3]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

print("Freeing up GPU memory from the Basic Transformer...")
# 1. Clear the previous model from VRAM to prevent Out-Of-Memory (OOM) errors
if 'model' in locals():
    del model, trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading mT5 tokenizer and pre-trained model...")
# 2. Load the pre-trained mT5-small model and tokenizer
model_name = "google/mt5-small"
mt5_tokenizer = AutoTokenizer.from_pretrained(model_name)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Tokenize with the mT5 task prefix
def preprocess_mt5(examples):
    inputs = ["translate English to French: " + ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    
    model_inputs = mt5_tokenizer(inputs, max_length=64, truncation=True, padding="max_length")
    labels = mt5_tokenizer(targets, max_length=64, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing data for mT5...")
mt5_tokenized_datasets = final_dataset.map(preprocess_mt5, batched=True)

# 4. Set up training arguments
mt5_training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=16, # Smaller batch size for the heavier mT5 model
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False, # mT5 can become mathematically unstable with fp16, so we keep it off
    save_strategy="epoch",
    report_to="none"
)

# 5. Train the model
mt5_trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=mt5_training_args,
    train_dataset=mt5_tokenized_datasets["train"],
    eval_dataset=mt5_tokenized_datasets["validation"],
    processing_class=mt5_tokenizer,
)

print("Starting mT5 fine-tuning! This will take a bit longer...")
import time
start_time_mt5 = time.time()

mt5_trainer.train()

mt5_training_time = time.time() - start_time_mt5
print(f"mT5 Fine-tuning completed in {mt5_training_time / 60:.2f} minutes.")

Freeing up GPU memory from the Basic Transformer...
Loading mT5 tokenizer and pre-trained model...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizing data for mT5...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mT5 Fine-tuning completed in 30.34 minutes.


In [ ]:
!pip install datasets transformers[torch] accelerate bitsandbytes huggingface_hub -q

from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login

# 1. Login to Hugging Face
login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

# 2. Re-fetch the manageable subset of the WMT14 dataset
print("Re-fetching dataset...")
wmt14_stream = load_dataset("wmt/wmt14", "fr-en", split="train", streaming=True)
subset_dataset = Dataset.from_list(list(wmt14_stream.take(25000)))

train_test_split = subset_dataset.train_test_split(test_size=0.2, seed=42)
val_test_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

final_dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': val_test_split['train'],
    'test': val_test_split['test']
})
print("Dataset and Environment Ready!")

Re-fetching dataset...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Dataset and Environment Ready!


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Loading LLaMA-3.2-1B in 4-bit precision...")
model_id = "meta-llama/Llama-3.2-1B"

# Use 4-bit quantization to shrink the model size for Kaggle's GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

llama_tokenizer = AutoTokenizer.from_pretrained(model_id)
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Creating a few-shot prompt function to guide the decoder-only architecture
def translate_llama(english_text):
    prompt = f"""Translate the following English text to French.
English: Hello, how are you?
French: Bonjour, comment allez-vous?
English: The weather is beautiful today.
French: Le temps est magnifique aujourd'hui.
English: {english_text}
French:"""
    
    inputs = llama_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = llama_model.generate(
        **inputs, 
        max_new_tokens=64, 
        do_sample=False,          # Switch to deterministic greedy decoding
        repetition_penalty=1.2,   # Penalize the model to break repetitive loops
        pad_token_id=llama_tokenizer.eos_token_id
    )
    
    # Added clean_up_tokenization_spaces=False to suppress the BPE warning
    full_text = llama_tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    # Split by "French:" and strictly isolate only the first generated line before any line breaks
    translation = full_text.split("French:")[-1].strip().split("\n")[0].strip()
    return translation

print("\nTesting LLaMA Translation:")
# Access the nested 'translation' dictionary before the language keys
test_sentence = final_dataset['test'][0]['translation']['en']
print(f"Original (English): {test_sentence}")
print(f"Actual (French): {final_dataset['test'][0]['translation']['fr']}")
print(f"LLaMA (French): {translate_llama(test_sentence)}")

Loading LLaMA-3.2-1B in 4-bit precision...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]


Testing LLaMA Translation:
Original (English): Quite the reverse, in fact. At present, there is a whole raft of companies which make a living out of them, and a decent one at that.
Actual (French): Il y a en ce moment toute une série d' entreprises qui en vivent et qui, grâce à cela, réalisent d' ailleurs un bénéfice non négligeable.
LLaMA (French): La situation est inverse en effet que des entreprises de toutes sortes qui font vivre d'entretenir et du moment qu'il y ait un certain nombre de personnes dont les intérêts sont diverses à la fois.


In [4]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Freeing up GPU memory from LLaMA...")
if 'llama_model' in locals():
    del llama_model
gc.collect()
torch.cuda.empty_cache()

print("Loading Gemma-2B-IT in 4-bit precision...")
# Switched to the Instruction-Tuned (-it) variant for task compliance
gemma_model_id = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def translate_gemma(english_text):
    # Using a more direct zero-shot prompt since the -it model understands instructions
    prompt = f"Translate the following English text to French. Do not include any explanations or conversational filler, just the direct translation.\nEnglish: {english_text}\nFrench:"
    
    inputs = gemma_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = gemma_model.generate(
        **inputs, 
        max_new_tokens=64, 
        do_sample=False, # Keep deterministic greedy decoding for accuracy
    )
    
    full_text = gemma_tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    translation = full_text.split("French:")[-1].strip().split("\n")[0].strip()
    return translation

print("\nTesting Gemma-IT Translation:")
test_sentence = final_dataset['test'][0]['translation']['en']
print(f"Original (English): {test_sentence}")
print(f"Gemma-IT (French): {translate_gemma(test_sentence)}")

Freeing up GPU memory from LLaMA...
Loading Gemma-2B-IT in 4-bit precision...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


Testing Gemma-IT Translation:
Original (English): Quite the reverse, in fact. At present, there is a whole raft of companies which make a living out of them, and a decent one at that.
Gemma-IT (French): Cela est une situation inverse, en effet. À présent, il y a une flotte de sociétés qui générent des revenus, et une entreprise qui offre une marge acceptable.


In [3]:
import torch
import gc
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Reloading LLaMA-3.2-1B for batch inference...")
model_id = "meta-llama/Llama-3.2-1B"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

llama_tokenizer = AutoTokenizer.from_pretrained(model_id)
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)

# Redefine the translation function in the current scope
def translate_llama(english_text):
    prompt = f"""Translate the following English text to French.
English: Hello, how are you?
French: Bonjour, comment allez-vous?
English: The weather is beautiful today.
French: Le temps est magnifique aujourd'hui.
English: {english_text}
French:"""
    
    inputs = llama_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = llama_model.generate(
        **inputs, 
        max_new_tokens=64, 
        do_sample=False,          # Switch to deterministic greedy decoding
        repetition_penalty=1.2,   # Penalize the model to break repetitive loops
        pad_token_id=llama_tokenizer.eos_token_id
    )
    
    # Suppress the BPE warning and strictly isolate the generated text
    full_text = llama_tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    translation = full_text.split("French:")[-1].strip().split("\n")[0].strip()
    return translation

# Extract the nested text for the full test set
references = [ex['translation']['fr'] for ex in final_dataset['test']]
english_texts = [ex['translation']['en'] for ex in final_dataset['test']]

llama_predictions = []
print("Generating full test set translations with LLaMA-3.2-1B...")

# Run without gradients to strictly prevent Kaggle VRAM leaks
with torch.no_grad():
    for text in tqdm(english_texts):
        llama_predictions.append(translate_llama(text))

# Free VRAM before moving to the next architectures
del llama_model, llama_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("LLaMA memory cleared.")

Reloading LLaMA-3.2-1B for batch inference...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Generating full test set translations with LLaMA-3.2-1B...


100%|██████████| 2500/2500 [1:05:44<00:00,  1.58s/it]


LLaMA memory cleared.


In [2]:
import gc
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Loading Gemma-2B-IT in 4-bit precision...")
gemma_model_id = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Redefine the translation function in the current scope
def translate_gemma(english_text):
    prompt = f"Translate the following English text to French. Do not include any explanations or conversational filler, just the direct translation.\nEnglish: {english_text}\nFrench:"
    
    inputs = gemma_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = gemma_model.generate(
        **inputs, 
        max_new_tokens=64, 
        do_sample=False, 
    )
    
    full_text = gemma_tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    translation = full_text.split("French:")[-1].strip().split("\n")[0].strip()
    return translation

# Extract the nested text for the full test set
references = [ex['translation']['fr'] for ex in final_dataset['test']]
english_texts = [ex['translation']['en'] for ex in final_dataset['test']]

gemma_predictions = []
print("Generating full test set translations with Gemma-2B-IT...")

with torch.no_grad():
    for text in tqdm(english_texts):
        gemma_predictions.append(translate_gemma(text))

# Free VRAM before moving to the next architectures
del gemma_model, gemma_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Gemma memory cleared.")

Loading Gemma-2B-IT in 4-bit precision...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Generating full test set translations with Gemma-2B-IT...


100%|██████████| 2500/2500 [59:01<00:00,  1.42s/it] 


Gemma memory cleared.


In [5]:
import os
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# Ensure the dataset text is available in the current scope
english_texts = [ex['translation']['en'] for ex in final_dataset['test']]

print("Loading trained Basic Transformer...")
basic_path = "./basic_transformer_results"
checkpoints = [d for d in os.listdir(basic_path) if d.startswith("checkpoint")]
latest_basic_ckpt = os.path.join(basic_path, max(checkpoints, key=lambda x: int(x.split("-")[-1])))

basic_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
basic_model = AutoModelForSeq2SeqLM.from_pretrained(latest_basic_ckpt).to("cuda")

basic_predictions = []
batch_size = 32

print("Generating translations with Basic Transformer...")
with torch.no_grad():
    for i in tqdm(range(0, len(english_texts), batch_size)):
        batch = english_texts[i:i+batch_size]
        inputs = basic_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")
        outputs = basic_model.generate(**inputs, max_length=64)
        basic_predictions.extend(basic_tokenizer.batch_decode(outputs, skip_special_tokens=True))

del basic_model, basic_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Basic Transformer memory cleared.")

Loading trained Basic Transformer...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Generating translations with Basic Transformer...


100%|██████████| 79/79 [00:38<00:00,  2.05it/s]


Basic Transformer memory cleared.


In [4]:
import os
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

print("Loading trained mT5-small...")
mt5_path = "./mt5_results"
checkpoints = [d for d in os.listdir(mt5_path) if d.startswith("checkpoint")]
latest_mt5_ckpt = os.path.join(mt5_path, max(checkpoints, key=lambda x: int(x.split("-")[-1])))

mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(latest_mt5_ckpt).to("cuda")

mt5_predictions = []
batch_size = 32

# Re-extract english_texts just to ensure it is in scope
english_texts = [ex['translation']['en'] for ex in final_dataset['test']]

print("Generating translations with mT5-small...")
with torch.no_grad():
    for i in tqdm(range(0, len(english_texts), batch_size)):
        # Re-apply the required task prefix for mT5
        batch = ["translate English to French: " + text for text in english_texts[i:i+batch_size]]
        inputs = mt5_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")
        outputs = mt5_model.generate(**inputs, max_length=64)
        mt5_predictions.extend(mt5_tokenizer.batch_decode(outputs, skip_special_tokens=True))

del mt5_model, mt5_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("mT5 memory cleared.")

Loading trained mT5-small...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generating translations with mT5-small...


100%|██████████| 79/79 [01:27<00:00,  1.11s/it]


mT5 memory cleared.


In [3]:
!pip install evaluate sacrebleu -q
import evaluate

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# Extract references directly from the dataset and format as a list of lists for the evaluate library
references = [ex['translation']['fr'] for ex in final_dataset['test']]
refs_formatted = [[ref] for ref in references]

def print_scores(model_name, predictions):
    b_score = bleu.compute(predictions=predictions, references=refs_formatted)["score"]
    c_score = chrf.compute(predictions=predictions, references=refs_formatted)["score"]
    print(f"--- {model_name} ---")
    print(f"BLEU: {b_score:.2f} | chrF: {c_score:.2f}\n")

print("Final Evaluation Scores:")
print_scores("LLaMA-3.2-1B (Few-Shot)", llama_predictions)
print_scores("Gemma-2B-IT (Zero-Shot)", gemma_predictions)
print_scores("Basic Transformer", basic_predictions)
print_scores("mT5-small", mt5_predictions)

Final Evaluation Scores:


NameError: name 'llama_predictions' is not defined